# Exploratory Data Analysis — Fake & Real News
Run from the `notebooks/` folder after placing `Fake.csv` and `True.csv` in `data/`.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, plotly.express as px
from collections import Counter
from src.preprocess import load_cleaned

df = load_cleaned()   # builds data/cleaned_news.csv on first run
df.head()

## 1. Class distribution

In [ ]:
counts = df.label.value_counts().reset_index(); counts.columns = ['label','count']
px.pie(counts, names='label', values='count', hole=.45, color='label',
       color_discrete_map={'FAKE':'#E74C3C','REAL':'#2ECC71'}, title='Class distribution')

## 2. Subjects and article length

In [ ]:
px.histogram(df, x='subject', color='label', barmode='group', title='Subjects by class')

In [ ]:
px.histogram(df[df.text_length < df.text_length.quantile(.99)], x='text_length', color='label',
             nbins=60, barmode='overlay', opacity=.65, title='Article length (words)')

## 3. Most common words

In [ ]:
def top_words(label, n=25):
    c = Counter()
    for t in df.loc[df.label == label, 'cleaned_text']: c.update(t.split())
    return pd.DataFrame(c.most_common(n), columns=['word','count'])

for lab in ['FAKE','REAL']:
    px.bar(top_words(lab).iloc[::-1], x='count', y='word', orientation='h', title=f'Top words — {lab}').show()

## 4. Fake vs Real comparison (distinctive words)

In [ ]:
import numpy as np
cf, cr = {}, {}
for lab, store in (('FAKE', cf), ('REAL', cr)):
    c = Counter()
    for t in df.loc[df.label == lab, 'cleaned_text']: c.update(t.split())
    store.update(c)
tf, tr = sum(cf.values()), sum(cr.values())
rows = [(w, np.log2((cf[w]/tf)/(cr[w]/tr))) for w in set(cf) & set(cr) if cf[w] + cr[w] >= 200]
diff = pd.DataFrame(rows, columns=['word','log2_ratio']).sort_values('log2_ratio')
px.bar(pd.concat([diff.head(15), diff.tail(15)]), x='log2_ratio', y='word', orientation='h',
       title='Words over-represented in REAL (left) vs FAKE (right)')